<a href="https://colab.research.google.com/github/oduntanfolake/FlyRank-ML-first-assignment-solution-/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oduntanfolake/FlyRank-ML-first-assignment-solution-/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
from google.colab import userdata
from huggingface_hub import HfApi

token = userdata.get("HF_TOKEN")

print("Token loaded:", token is not None)
print("Token length:", len(token) if token else 0)

api = HfApi(token=token)
me = api.whoami()

print("Hugging Face login:", me["name"])


Token loaded: True
Token length: 37
Hugging Face login: Fbeva


In [4]:
import duckdb

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [token]
)

print("DuckDB → Hugging Face connection ready.")

DuckDB → Hugging Face connection ready.


In [5]:
import duckdb

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN ?)",
    [token]
)

print("DuckDB connected to Hugging Face.")

DuckDB connected to Hugging Face.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: Random Forest. A Random Forest is appropriate because this lane is focused on ranking pages by engagement opportunity using multiple observed search and engagement signals. Random Forest can capture nonlinear relationships and interactions between signals such as impressions, clicks, CTR, and performance changes without requiring a complex hand-written rule. The model will be used to produce an opportunity score for prioritization, not as an automatic decision about whether a page should be changed.I use a small number of trees for this baseline so the experiment remains computationally practical

The model will be compared against the Week-4 rule-based baseline using the same data, split, and evaluation approach.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1 — Check the columns available for modeling

model_columns = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "sessions_ai",
    "scroll_events"
]

available_model_columns = [
    col for col in model_columns
    if col in con.sql("""
        SELECT column_name
        FROM (
            DESCRIBE SELECT *
            FROM read_parquet(
                'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
            )
        )
    """).df()["column_name"].tolist()
]

print("Available modeling columns:")
print(available_model_columns)


Available modeling columns:
['gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'sessions_ai', 'scroll_events']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a grouped split by client_hash_id so that content from the same client does not appear in both the training and evaluation sets. This gives a more honest test of whether the model can rank engagement opportunities for clients it did not learn from.

The split will be performed before model fitting, and the same evaluation data will be used to compare the Random Forest against the Week-4 baseline. No future-month performance will be used as a feature.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# SECTION 2 — Split design

from sklearn.model_selection import GroupShuffleSplit

march_path = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'

# Create a compact page-level March dataset.
# We aggregate daily observations to one row per client + content page.
df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_sum_position) AS avg_position,
    SUM(sessions_ai) AS sessions_ai,
    SUM(scroll_events) AS scroll_events
FROM read_parquet('{march_path}')
GROUP BY client_hash_id, content_hash_id
""").df()

print("Page-level rows:", len(df))
print("Unique clients:", df["client_hash_id"].nunique())
print("Unique content pages:", df["content_hash_id"].nunique())

# Client-grouped split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["client_hash_id"])
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

print("\nTrain rows:", len(train))
print("Test rows:", len(test))
print("Train clients:", train["client_hash_id"].nunique())
print("Test clients:", test["client_hash_id"].nunique())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Page-level rows: 331437
Unique clients: 55
Unique content pages: 331437

Train rows: 300880
Test rows: 30557
Train clients: 44
Test clients: 11


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I trained a Random Forest model using the March 2026 development data and evaluated it on a client-grouped test split. The model was compared with the Week-4 baseline using Precision@50, so both approaches were evaluated on the same test set and with the same ranking metric.

The Week-4 baseline achieved a Precision@50 of 0.00, while the Random Forest achieved 0.70. Under the engagement-opportunity proxy used for this analysis, this means that 0 of the baseline's top 50 pages and 35 of the model's top 50 pages were identified as opportunities.

The 35 of 50 figure comes directly from 0.70 × 50 = 35. This result indicates that the Random Forest produced a substantially more useful top-50 ranking than the simple baseline on this test split.

However, this should be treated as directional evidence rather than proof of real-world performance. The opportunity measure is a proxy based on observed engagement signals rather than an independently validated business outcome. Therefore, the model's Precision@50 should be interpreted as performance against this proxy, not as proof that 70% of the pages have genuine engagement problems.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# SECTION 3 — Train + compare against baseline

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import pandas as pd
import numpy as np
features = [
    "gsc_impressions",
    "gsc_clicks",
    "avg_position",
    "sessions_ai",
    "scroll_events"
]

train["engagement_proxy"] = (
    train["gsc_impressions"].fillna(0)
    / (train["gsc_clicks"].fillna(0) + 1)
)

test["engagement_proxy"] = (
    test["gsc_impressions"].fillna(0)
    / (test["gsc_clicks"].fillna(0) + 1)
)

X_train = train[features].fillna(0)
y_train = train["engagement_proxy"]

X_test = test[features].fillna(0)
y_test = test["engagement_proxy"]

model = RandomForestRegressor(
    n_estimators=30,
    random_state=42,
    n_jobs=-1,
    max_depth=10
)

model.fit(X_train, y_train)

model_pred = model.predict(X_test)

baseline_pred = X_test["gsc_impressions"].values

model_mae = mean_absolute_error(y_test, model_pred)
baseline_mae = mean_absolute_error(y_test, baseline_pred)

results = pd.DataFrame({
    "method": ["Week-4 baseline", "Random Forest"],
    "MAE": [baseline_mae, model_mae]
})

print(results)

            method         MAE
0  Week-4 baseline  651.190952
1    Random Forest    5.471564


In [9]:
# Compare the Week-4 baseline and Random Forest using Precision@50

def precision_at_50(scores, actual_opportunity):
    ranking = pd.DataFrame({
        "score": scores,
        "actual": actual_opportunity
    }).sort_values("score", ascending=False)

    top_50 = ranking.head(50)
    return top_50["actual"].mean()

test["actual_opportunity"] = (
    (test["gsc_impressions"].fillna(0) > 0) &
    (test["gsc_clicks"].fillna(0) == 0)
).astype(int)


baseline_scores = test["gsc_impressions"].fillna(0)


model_scores = model_pred


baseline_p50 = precision_at_50(
    baseline_scores,
    test["actual_opportunity"]
)

model_p50 = precision_at_50(
    model_scores,
    test["actual_opportunity"]
)


comparison = pd.DataFrame({
    "method": ["Week-4 baseline", "Random Forest"],
    "Precision@50": [baseline_p50, model_p50]
})

print(comparison)

            method  Precision@50
0  Week-4 baseline           0.0
1    Random Forest           0.7


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*


The model was evaluated on the already-created test set rather than reloading the full warehouse. This keeps the error analysis focused on the held-out predictions and avoids unnecessarily reprocessing the full 9.8-million-row dataset.

The Random Forest placed substantially more weight on `gsc_impressions` (0.734) and `gsc_clicks` (0.204), with smaller contributions from `avg_position`, `scroll_events`, and `sessions_ai`. This suggests that search visibility and click activity were the strongest signals used by the model for ranking engagement opportunities.

The top-ranked results show that the model can identify pages with high impressions and low or zero clicks as potential engagement opportunities. However, some highly ranked pages were not marked as opportunities under the proxy, showing that the model can still make incorrect selections.

The Precision@50 result of 0.70 means that 35 of the model's top 50 ranked pages were relevant under the proxy used. This is an observed evaluation result, not proof that the model will identify 70% of real-world engagement problems.

The model therefore provides decision support: it prioritizes pages for investigation, while a website manager still needs to review the pages and determine what action is appropriate.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# SECTION 4 — Errors and interpretation

# Feature importance
importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("Feature importance:")
print(importance.to_string(index=False))


# Inspect the model's top 10 predictions
error_check = test[[
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks"
]].copy()

error_check["model_score"] = model_pred
error_check["actual_opportunity"] = test["actual_opportunity"].values

error_check = error_check.sort_values(
    "model_score",
    ascending=False
)

print("\nTop 10 model-ranked pages:")
print(error_check.head(10).to_string(index=False))

Feature importance:
        feature  importance
gsc_impressions    0.734085
     gsc_clicks    0.203938
   avg_position    0.056425
  scroll_events    0.003697
    sessions_ai    0.001855

Top 10 model-ranked pages:
         client_hash_id          content_hash_id  gsc_impressions  gsc_clicks  model_score  actual_opportunity
client_e547b89c05043229 content_713b157e9c77690a          24908.0         0.0 22596.888889                   1
client_e547b89c05043229 content_545bb6cc7081ded3         122905.0       287.0 15197.583480                   0
client_c182d11e4862a37d content_d2eb49b1f5f3fa34          14482.0         0.0 14755.308333                   1
client_e547b89c05043229 content_4002467a580a7f98          11973.0         0.0 12173.974762                   1
client_e547b89c05043229 content_dc91779c3d085398          25625.0         1.0 11886.312501                   0
client_e547b89c05043229 content_0e2e4d3ab02abc1a          11187.0         0.0 11232.362593                   1
client_

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.